# Enterprise Telecom Deal Pricing, Hardware Credit & Commercial SOW Risk Engine
**Commercial Finance & Deal Desk Analytics | Portfolio Proof-of-Work (`telecom-commercial-pricing-engine`)**

This notebook implements an institutional-grade B2B Deal Pricing and Risk Management Model designed for an enterprise telecommunications Commercial Analyst function. The model evaluates multi-tier enterprise telecommunications proposals (300+ mobile connections and regional fixed fiber links) against benchmark rate cards. It calculates Total Contract Value (TCV), Gross and Net Profit Margins, models $25,000 upfront Hardware Credit Fund amortization schedules, stress-tests margin resilience under roaming usage spikes and connection churn, and executes an automated Statement of Work (SOW) Commercial Risk Assessment. Finally, it exports a fully formatted, multi-tab Excel model (`Telecom_Enterprise_Deal_Model.xlsx`) and generates interactive C-suite executive briefing charts.

In [1]:
# Installs and Imports
import sys
import subprocess

# Ensure required packages are installed
required_packages = ['plotly', 'openpyxl', 'pandas', 'numpy', 'numpy_financial', 'matplotlib', 'seaborn']
for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import numpy_financial as npf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print("✓ All libraries successfully loaded and ready.")

✓ All libraries successfully loaded and ready.


In [2]:
# Configuration Engine: Telecom Benchmark Rate Cards, Direct COGS, and Client Profile

# 1. Standard Enterprise Rate Cards & Wholesale COGS (Monthly per unit in NZD)
RATE_CARD_DATABASE = {
    'Mobile_Exec_Unlimited': {'list_price': 85.00, 'direct_cogs': 22.00, 'category': 'Mobile'},
    'Mobile_Field_Standard':  {'list_price': 65.00, 'direct_cogs': 15.00, 'category': 'Mobile'},
    'Mobile_Desk_Basic':     {'list_price': 45.00, 'direct_cogs': 10.00, 'category': 'Mobile'},
    'Fixed_Fiber_1Gbps':     {'list_price': 250.00, 'direct_cogs': 85.00, 'category': 'Fixed Broadband'}
}

# 2. Target Client Profile: Enterprise Account (300 Mobile Connections + 8 Fiber Sites)
CLIENT_PROFILE = {
    'client_name': 'Aotearoa Logistics Group',
    'industry': 'Transport & Supply Chain',
    'quantities': {
        'Mobile_Exec_Unlimited': 50,
        'Mobile_Field_Standard': 150,
        'Mobile_Desk_Basic': 100,
        'Fixed_Fiber_1Gbps': 8
    }
}

# 3. Commercial Deal Governance Rules
GOVERNANCE_TARGETS = {
    'min_gross_margin_pct': 45.0,  # Target minimum Gross Margin %
    'min_net_margin_pct': 30.0,    # Target minimum Net Margin % (after HW Fund)
    'max_payback_months': 12       # Maximum acceptable payback period for HW Fund
}

df_rate_cards = pd.DataFrame.from_dict(RATE_CARD_DATABASE, orient='index')
df_rate_cards['Target_Qty'] = df_rate_cards.index.map(CLIENT_PROFILE['quantities'])
df_rate_cards['Monthly_List_Revenue'] = df_rate_cards['list_price'] * df_rate_cards['Target_Qty']
df_rate_cards['Monthly_Total_COGS'] = df_rate_cards['direct_cogs'] * df_rate_cards['Target_Qty']

print("=== TELECOM RATE CARD & CLIENT PROFILE CONFIGURATION ===")
print(df_rate_cards[['category', 'Target_Qty', 'list_price', 'direct_cogs', 'Monthly_List_Revenue']])
print(f"\nTotal Monthly List Revenue: ${df_rate_cards['Monthly_List_Revenue'].sum():,.2f} NZD")

=== TELECOM RATE CARD & CLIENT PROFILE CONFIGURATION ===
                              category  Target_Qty  list_price  direct_cogs  \
Mobile_Exec_Unlimited           Mobile          50        85.0         22.0   
Mobile_Field_Standard           Mobile         150        65.0         15.0   
Mobile_Desk_Basic               Mobile         100        45.0         10.0   
Fixed_Fiber_1Gbps      Fixed Broadband           8       250.0         85.0   

                       Monthly_List_Revenue  
Mobile_Exec_Unlimited                4250.0  
Mobile_Field_Standard                9750.0  
Mobile_Desk_Basic                    4500.0  
Fixed_Fiber_1Gbps                    2000.0  

Total Monthly List Revenue: $20,500.00 NZD


In [3]:
# B2B Financial Modelling & Deal Engine Class

class TelecomDealEngine:
    def __init__(self, rate_card_df, client_profile, mobile_discount_pct=0.15, fiber_discount_pct=0.10,
                 hardware_fund=25000.0, term_months=24, roaming_cost_surge=0.0):

        self.df = rate_card_df.copy()
        self.client_name = client_profile['client_name']
        self.mobile_discount = mobile_discount_pct
        self.fiber_discount = fiber_discount_pct
        self.hardware_fund = hardware_fund
        self.term_months = term_months
        self.roaming_cost_surge = roaming_cost_surge

        self._calculate_deal()

    def _calculate_deal(self):
        # Apply tier-based discounting
        discount_map = {
            'Mobile': self.mobile_discount,
            'Fixed Broadband': self.fiber_discount
        }

        self.df['Discount_Pct'] = self.df['category'].map(discount_map)
        self.df['Discounted_Price'] = self.df['list_price'] * (1 - self.df['Discount_Pct'])
        self.df['Monthly_Net_Revenue'] = self.df['Discounted_Price'] * self.df['Target_Qty']

        # Apply Roaming Surge to Direct COGS if scenario calls for it
        self.df['Adjusted_COGS'] = self.df['Monthly_Total_COGS'] * (1 + self.roaming_cost_surge)
        self.df['Monthly_Gross_Margin'] = self.df['Monthly_Net_Revenue'] - self.df['Adjusted_COGS']

        # Summary Level Metrics
        self.monthly_list_rev = self.df['Monthly_List_Revenue'].sum()
        self.monthly_net_rev = self.df['Monthly_Net_Revenue'].sum()
        self.monthly_cogs = self.df['Adjusted_COGS'].sum()
        self.monthly_gross_margin = self.df['Monthly_Gross_Margin'].sum()

        # Rate Variance
        self.rate_variance_mrr = self.monthly_list_rev - self.monthly_net_rev
        self.rate_variance_pct = (self.rate_variance_mrr / self.monthly_list_rev) * 100

        # Hardware Amortization & Net Margin
        self.monthly_hw_amort = self.hardware_fund / self.term_months
        self.monthly_net_profit = self.monthly_gross_margin - self.monthly_hw_amort

        # Contract Totals
        self.tcv = self.monthly_net_rev * self.term_months
        self.acv = self.tcv / (self.term_months / 12)
        self.total_cogs = self.monthly_cogs * self.term_months
        self.total_gross_margin = self.monthly_gross_margin * self.term_months
        self.total_net_profit = self.total_gross_margin - self.hardware_fund

        # Percentage Returns
        self.gross_margin_pct = (self.total_gross_margin / self.tcv) * 100
        self.net_margin_pct = (self.total_net_profit / self.tcv) * 100

        # Monthly Schedule Engine
        self.schedule = self._build_monthly_schedule()

        # Calculate Hardware Payback Month
        payback_rows = self.schedule[self.schedule['Cumulative_Net_Profit'] >= 0]
        self.payback_month = int(payback_rows.iloc[0]['Month']) if not payback_rows.empty else self.term_months + 1

    def _build_monthly_schedule(self):
        schedule_data = []
        cum_gross = 0.0
        cum_net = -self.hardware_fund  # Starts negative due to upfront hardware outlay

        for m in range(1, self.term_months + 1):
            cum_gross += self.monthly_gross_margin
            cum_net += self.monthly_gross_margin

            schedule_data.append({
                'Month': m,
                'Net_Revenue': self.monthly_net_rev,
                'Direct_COGS': self.monthly_cogs,
                'Gross_Margin': self.monthly_gross_margin,
                'HW_Amortization': self.monthly_hw_amort,
                'Net_Profit': self.monthly_net_profit,
                'Cumulative_Gross_Margin': cum_gross,
                'Cumulative_Net_Profit': cum_net
            })

        return pd.DataFrame(schedule_data)

# Run Base Deal
base_deal = TelecomDealEngine(
    rate_card_df=df_rate_cards,
    client_profile=CLIENT_PROFILE,
    mobile_discount_pct=0.15,
    fiber_discount_pct=0.10,
    hardware_fund=25000.0,
    term_months=24
)

print(f"=== BASE DEAL FINANCIAL SUMMARY ({base_deal.client_name}) ===")
print(f"Total Contract Value (TCV):  ${base_deal.tcv:,.2f} NZD")
print(f"Annual Contract Value (ACV): ${base_deal.acv:,.2f} NZD")
print(f"Rate Card Discount Variance: {base_deal.rate_variance_pct:.2f}% (${base_deal.rate_variance_mrr:,.2f}/mo)")
print(f"Gross Margin %:              {base_deal.gross_margin_pct:.2f}% (${base_deal.total_gross_margin:,.2f})")
print(f"Net Margin % (Post-HW Fund): {base_deal.net_margin_pct:.2f}% (${base_deal.total_net_profit:,.2f})")
print(f"Hardware Fund Payback:       Month {base_deal.payback_month} of {base_deal.term_months}")

=== BASE DEAL FINANCIAL SUMMARY (Aotearoa Logistics Group) ===
Total Contract Value (TCV):  $420,600.00 NZD
Annual Contract Value (ACV): $210,300.00 NZD
Rate Card Discount Variance: 14.51% ($2,975.00/mo)
Gross Margin %:              71.30% ($299,880.00)
Net Margin % (Post-HW Fund): 65.35% ($274,880.00)
Hardware Fund Payback:       Month 3 of 24


In [4]:
# Scenario Stress-Testing & Sensitivity Engine

def run_commercial_scenarios(rate_card_df, client_profile):
    scenarios = {}

    # 1. Base Case (15% Mobile, 10% Fiber discount, standard usage)
    scenarios['Base Case'] = TelecomDealEngine(
        rate_card_df, client_profile,
        mobile_discount_pct=0.15, fiber_discount_pct=0.10,
        hardware_fund=25000.0, term_months=24, roaming_cost_surge=0.0
    )

    # 2. High Roaming Usage Spike (+30% surge in network roaming COGS)
    scenarios['High Roaming Exposure'] = TelecomDealEngine(
        rate_card_df, client_profile,
        mobile_discount_pct=0.15, fiber_discount_pct=0.10,
        hardware_fund=25000.0, term_months=24, roaming_cost_surge=0.30
    )

    # 3. Aggressive Sales Discounting (25% Mobile, 15% Fiber discount + $30k HW Fund)
    scenarios['Aggressive Discounting'] = TelecomDealEngine(
        rate_card_df, client_profile,
        mobile_discount_pct=0.25, fiber_discount_pct=0.15,
        hardware_fund=30000.0, term_months=24, roaming_cost_surge=0.0
    )

    summary_list = []
    for name, engine in scenarios.items():
        summary_list.append({
            'Scenario': name,
            'TCV ($)': engine.tcv,
            'Rate Variance %': engine.rate_variance_pct,
            'Gross Margin %': engine.gross_margin_pct,
            'Net Margin %': engine.net_margin_pct,
            'Payback Month': engine.payback_month,
            'Passes Min Gross Target': engine.gross_margin_pct >= GOVERNANCE_TARGETS['min_gross_margin_pct'],
            'Passes Payback Target': engine.payback_month <= GOVERNANCE_TARGETS['max_payback_months']
        })

    return pd.DataFrame(summary_list), scenarios

df_scenarios, scenario_engines = run_commercial_scenarios(df_rate_cards, CLIENT_PROFILE)

print("=== SCENARIO STRESS-TEST COMPARISON ===")
print(df_scenarios.to_string(index=False))

=== SCENARIO STRESS-TEST COMPARISON ===
              Scenario  TCV ($)  Rate Variance %  Gross Margin %  Net Margin %  Payback Month  Passes Min Gross Target  Passes Payback Target
             Base Case 420600.0        14.512195       71.298146     65.354256              3                     True                   True
 High Roaming Exposure 420600.0        14.512195       62.687589     56.743699              3                     True                   True
Aggressive Discounting 373800.0        24.024390       67.704655     59.678973              3                     True                   True


In [5]:
# Statement of Work (SOW) Commercial Risk Assessment Engine

class SOWRiskEvaluator:
    def __init__(self, contract_clauses):
        self.clauses = contract_clauses
        self.risk_score = 0
        self.findings = []
        self._evaluate_risk()

    def _evaluate_risk(self):
        # Rule 1: SLA Penalty Cap
        if self.clauses.get('sla_penalty_cap_pct', 100) > 15:
            self.risk_score += 25
            self.findings.append("HIGH RISK: SLA penalty cap exceeds maximum 15% monthly fee threshold.")
        else:
            self.findings.append("LOW RISK: SLA penalty cap governed within allowable limit.")

        # Rule 2: Annual CPI Indexation
        if not self.clauses.get('cpi_indexation_included', False):
            self.risk_score += 20
            self.findings.append("MEDIUM RISK: Missing annual CPI rate adjustment clause.")
        else:
            self.findings.append("LOW RISK: Annual CPI indexation clause present.")

        # Rule 3: Hardware Credit Expiry Policy
        if self.clauses.get('hw_credit_expiry_months', 0) > 24:
            self.risk_score += 20
            self.findings.append("MEDIUM RISK: Hardware credit rollover exceeds 24 months.")
        else:
            self.findings.append("LOW RISK: Hardware credit expires within contract term.")

        # Rule 4: Early Termination Liquidated Damages
        if not self.clauses.get('full_early_termination_recovery', False):
            self.risk_score += 25
            self.findings.append("HIGH RISK: SOW lacks full unamortized hardware credit recovery on exit.")
        else:
            self.findings.append("LOW RISK: Full hardware credit clawback enforceable upon early exit.")

        # Rule 5: Billing Start Trigger
        if self.clauses.get('billing_start_trigger') == 'upon_site_deployment':
            self.risk_score += 10
            self.findings.append("LOW-MED RISK: Billing starts on site deployment rather than network activation.")
        else:
            self.findings.append("LOW RISK: Billing starts immediately upon network provisioning.")

    def get_risk_tier(self):
        if self.risk_score <= 20:
            return "LOW RISK (Approved)"
        elif self.risk_score <= 45:
            return "MEDIUM RISK (Conditional Approval)"
        else:
            return "HIGH RISK (Requires C-Suite Sign-off)"

# Test SOW Contract Sample
sow_clause_inputs = {
    'sla_penalty_cap_pct': 10,
    'cpi_indexation_included': False,
    'hw_credit_expiry_months': 24,
    'full_early_termination_recovery': True,
    'billing_start_trigger': 'upon_provisioning'
}

sow_evaluator = SOWRiskEvaluator(sow_clause_inputs)

print("=== SOW COMMERCIAL RISK EVALUATION ===")
print(f"Overall Risk Score: {sow_evaluator.risk_score} / 100")
print(f"Risk Tier Assessment: {sow_evaluator.get_risk_tier()}\n")
print("Detailed Audit Findings:")
for f in sow_evaluator.findings:
    print(f" - {f}")

=== SOW COMMERCIAL RISK EVALUATION ===
Overall Risk Score: 20 / 100
Risk Tier Assessment: LOW RISK (Approved)

Detailed Audit Findings:
 - LOW RISK: SLA penalty cap governed within allowable limit.
 - MEDIUM RISK: Missing annual CPI rate adjustment clause.
 - LOW RISK: Hardware credit expires within contract term.
 - LOW RISK: Full hardware credit clawback enforceable upon early exit.
 - LOW RISK: Billing starts immediately upon network provisioning.


In [6]:
# Dynamic Visualizations Engine (Plotly Interactive Charts)

# 1. Deal Margin Waterfall Chart
base_e = scenario_engines['Base Case']

fig_waterfall = go.Figure(go.Waterfall(
    name = "Base Deal Margin",
    orientation = "v",
    measure = ["relative", "relative", "total", "relative", "relative", "total"],
    x = ["List Revenue", "Sales Discounts", "Net Revenue", "Direct COGS", "HW Fund Outlay", "Net Profit"],
    textposition = "outside",
    text = [f"${base_e.monthly_list_rev*24:,.0f}", f"-${base_e.rate_variance_mrr*24:,.0f}",
            f"${base_e.tcv:,.0f}", f"-${base_e.total_cogs:,.0f}", f"-${base_e.hardware_fund:,.0f}",
            f"${base_e.total_net_profit:,.0f}"],
    y = [base_e.monthly_list_rev*24, -base_e.rate_variance_mrr*24, base_e.tcv,
         -base_e.total_cogs, -base_e.hardware_fund, base_e.total_net_profit],
    connector = {"line":{"color":"rgb(63, 63, 63)"}},
    decreasing = {"marker":{"color":"#E60000"}},
    increasing = {"marker":{"color":"#2A2A2A"}},
    totals = {"marker":{"color":"#002060"}}
))

fig_waterfall.update_layout(
    title = "<b>Enterprise Telecom B2B Deal Waterfall: From List Revenue to Net Profit (24 Month TCV)</b>",
    yaxis_title = "NZD ($)",
    template = "plotly_white",
    height = 500
)

# 2. Hardware Credit Payback Trajectory Chart
sched = base_e.schedule
fig_payback = go.Figure()

fig_payback.add_trace(go.Scatter(
    x=sched['Month'], y=sched['Cumulative_Net_Profit'],
    mode='lines+markers', name='Cumulative Net Profit ($)',
    line=dict(color='#002060', width=3),
    marker=dict(size=6)
))

fig_payback.add_shape(
    type="line", x0=1, y0=0, x1=24, y1=0,
    line=dict(color="red", width=2, dash="dash")
)

fig_payback.add_annotation(
    x=base_e.payback_month, y=0,
    text=f" Payback Point: Month {base_e.payback_month}",
    showarrow=True, arrowhead=2, ax=40, ay=-40,
    font=dict(color="#002060", size=12, family="Arial")
)

fig_payback.update_layout(
    title = "<b>Hardware Credit Fund Amortization & Profit Breakeven Curve</b>",
    xaxis_title = "Contract Month",
    yaxis_title = "Cumulative Net Profit / Loss (NZD)",
    template = "plotly_white",
    height = 450
)

fig_waterfall.show()
fig_payback.show()

In [7]:
# OpenPyXL Excel Model Generator (Generates 'Telecom_Enterprise_Deal_Model.xlsx')

def generate_formatted_excel_model(deal_engine, sow_eval, filename="Telecom_Enterprise_Deal_Model.xlsx"):
    wb = openpyxl.Workbook()

    # Color Palette - Executive Theme
    NAVY_HEADER = "002060"
    GRAY_LIGHT = "F2F2F2"
    WHITE_TEXT = "FFFFFF"

    font_title = Font(name="Calibri", size=16, bold=True, color=NAVY_HEADER)
    font_header = Font(name="Calibri", size=11, bold=True, color=WHITE_TEXT)
    font_bold = Font(name="Calibri", size=11, bold=True)

    fill_header = PatternFill(start_color=NAVY_HEADER, end_color=NAVY_HEADER, fill_type="solid")

    thin_border = Border(left=Side(style='thin', color='D9D9D9'),
                         right=Side(style='thin', color='D9D9D9'),
                         top=Side(style='thin', color='D9D9D9'),
                         bottom=Side(style='thin', color='D9D9D9'))

    # TAB 1: Executive Deal Summary
    ws1 = wb.active
    ws1.title = "Executive Deal Summary"

    ws1['A1'] = "ENTERPRISE TELECOM B2B DEAL APPROVAL MODEL"
    ws1['A1'].font = font_title
    ws1['A2'] = f"Client: {deal_engine.client_name} | Contract Term: {deal_engine.term_months} Months"

    summary_data = [
        ["Key Metric", "Value (NZD / %)", "Governance Target", "Status"],
        ["Total Contract Value (TCV)", deal_engine.tcv, "-", "INFORMATIONAL"],
        ["Annual Contract Value (ACV)", deal_engine.acv, "-", "INFORMATIONAL"],
        ["Rate Card Discount Variance", f"{deal_engine.rate_variance_pct:.2f}%", "< 20.0%", "PASS"],
        ["Total Direct Network COGS", deal_engine.total_cogs, "-", "COST"],
        ["Hardware Credit Fund", deal_engine.hardware_fund, "$25,000 Cap", "APPROVED"],
        ["Gross Profit Margin %", f"{deal_engine.gross_margin_pct:.2f}%", f">= {GOVERNANCE_TARGETS['min_gross_margin_pct']}%", "PASS" if deal_engine.gross_margin_pct >= 45 else "FLAG"],
        ["Net Profit Margin %", f"{deal_engine.net_margin_pct:.2f}%", f">= {GOVERNANCE_TARGETS['min_net_margin_pct']}%", "PASS" if deal_engine.net_margin_pct >= 30 else "FLAG"],
        ["Hardware Fund Payback Month", f"Month {deal_engine.payback_month}", f"<= Month {GOVERNANCE_TARGETS['max_payback_months']}", "PASS" if deal_engine.payback_month <= 12 else "FLAG"],
        ["SOW Risk Assessment Score", f"{sow_eval.risk_score} / 100", "Low Risk Tier", sow_eval.get_risk_tier()]
    ]

    for r_idx, row in enumerate(summary_data, start=4):
        for c_idx, val in enumerate(row, start=1):
            cell = ws1.cell(row=r_idx, column=c_idx, value=val)
            cell.border = thin_border
            if r_idx == 4:
                cell.font = font_header
                cell.fill = fill_header
                cell.alignment = Alignment(horizontal="center")
            elif c_idx == 1:
                cell.font = font_bold

    ws1.column_dimensions['A'].width = 30
    ws1.column_dimensions['B'].width = 22
    ws1.column_dimensions['C'].width = 20
    ws1.column_dimensions['D'].width = 35

    # TAB 2: Monthly Financial Amortization Schedule
    ws2 = wb.create_sheet(title="Financial Schedule")

    ws2.append(["Month", "Net Revenue ($)", "Direct COGS ($)", "Gross Margin ($)", "HW Amortization ($)", "Net Profit ($)", "Cum. Net Profit ($)"])
    for cell in ws2[1]:
        cell.font = font_header
        cell.fill = fill_header

    for _, row in deal_engine.schedule.iterrows():
        ws2.append([
            int(row['Month']), row['Net_Revenue'], row['Direct_COGS'],
            row['Gross_Margin'], row['HW_Amortization'], row['Net_Profit'],
            row['Cumulative_Net_Profit']
        ])

    for row in ws2.iter_rows(min_row=2, max_col=7, max_row=ws2.max_row):
        for cell in row:
            cell.border = thin_border
            if cell.column > 1:
                cell.number_format = "$#,##0.00"

    wb.save(filename)
    print(f"✓ Excel Workbook successfully generated and saved as '{filename}'")

generate_formatted_excel_model(base_deal, sow_evaluator)

✓ Excel Workbook successfully generated and saved as 'Telecom_Enterprise_Deal_Model.xlsx'


In [8]:
# C-Suite Executive Briefing Card Output

def display_executive_briefing_card(deal_engine, sow_eval):
    overall_status = "APPROVED" if (deal_engine.gross_margin_pct >= 45 and sow_eval.risk_score <= 30) else "CONDITIONAL APPROVAL"

    print("="*65)
    print("      TELECOM COMMERCIAL DEAL DESK - EXECUTIVE BRIEFING CARD      ")
    print("="*65)
    print(f" Client Name:          {deal_engine.client_name}")
    print(f" Contract Duration:    {deal_engine.term_months} Months")
    print(f" Total Connections:    300 Mobile Fleets + 8 Regional Fiber Sites")
    print("-"*65)
    print(f" Total Contract Value: ${deal_engine.tcv:,.2f} NZD")
    print(f" Annual Contract Value:${deal_engine.acv:,.2f} NZD")
    print(f" Monthly Net Revenue:  ${deal_engine.monthly_net_rev:,.2f} NZD")
    print(f" Rate Card Variance:   -{deal_engine.rate_variance_pct:.2f}% (${deal_engine.rate_variance_mrr:,.2f}/mo discount)")
    print("-"*65)
    print(f" Gross Profit Margin:  {deal_engine.gross_margin_pct:.2f}% (${deal_engine.total_gross_margin:,.2f}) [Target: >=45%]")
    print(f" Net Profit Margin:    {deal_engine.net_margin_pct:.2f}% (${deal_engine.total_net_profit:,.2f})")
    print(f" Hardware Fund Outlay: ${deal_engine.hardware_fund:,.2f} NZD")
    print(f" Hardware Payback:     Month {deal_engine.payback_month} [Target: <=12 Months]")
    print("-"*65)
    print(f" SOW Risk Score:       {sow_eval.risk_score} / 100 ({sow_eval.get_risk_tier()})")
    print(f" FINAL RECOMMENDATION:  {overall_status}")
    print("="*65)

display_executive_briefing_card(base_deal, sow_evaluator)

      TELECOM COMMERCIAL DEAL DESK - EXECUTIVE BRIEFING CARD      
 Client Name:          Aotearoa Logistics Group
 Contract Duration:    24 Months
 Total Connections:    300 Mobile Fleets + 8 Regional Fiber Sites
-----------------------------------------------------------------
 Total Contract Value: $420,600.00 NZD
 Annual Contract Value:$210,300.00 NZD
 Monthly Net Revenue:  $17,525.00 NZD
 Rate Card Variance:   -14.51% ($2,975.00/mo discount)
-----------------------------------------------------------------
 Gross Profit Margin:  71.30% ($299,880.00) [Target: >=45%]
 Net Profit Margin:    65.35% ($274,880.00)
 Hardware Fund Outlay: $25,000.00 NZD
 Hardware Payback:     Month 3 [Target: <=12 Months]
-----------------------------------------------------------------
 SOW Risk Score:       20 / 100 (LOW RISK (Approved))
 FINAL RECOMMENDATION:  APPROVED


### Summary & Portfolio Impact

**What This Notebook Achieved:**
1. **Commercial Deal Structuring:** Built an end-to-end B2B pricing model evaluating a 300-connection enterprise account across standard mobile and fixed rate cards.
2. **Hardware Credit & Margin Amortization:** Modeled a $25,000 upfront hardware credit fund, establishing exact payback month timelines and gross/net profit margin waterfalls.
3. **Scenario Stress-Testing:** Tested deal resilience against high international roaming usage surges (+30%) and aggressive discounting.
4. **SOW Risk Governance:** Built an automated checklist scanner evaluating 5 critical commercial contract terms (SLA caps, CPI indexation, early exit penalties).
5. **C-Suite Automation:** Exported a fully formatted multi-tab Excel model (`Telecom_Enterprise_Deal_Model.xlsx`) and generated interactive Plotly visualizations.

---

### Resume-Ready Portfolio Bullet Point
> **B2B Enterprise Telco Pricing & Risk Engine (Python/Excel)** | [GitHub Link]
> * Engineered an automated enterprise deal engine evaluating multi-tier pricing, $25k hardware credit payback schedules, and margin waterfalls for 300+ connection accounts; built scenario stress-testing and an SOW risk evaluation matrix exporting C-suite approval decks and formatted Excel models.